# PlantMetWiki figures

Generates publication-quality figures for PlantMetWiki from SPARQL queries against the Virtuoso endpoint.

**Workflow:**
1. Run SPARQL queries → save CSVs to `notebooks/figures/`
2. Generate figures from CSVs → save PDF + SVG to `notebooks/figures/`

The `notebooks/figures/` directory also contains **reference CSVs** exported from a previous run via Snorql — useful for comparing results across builds.

**Kernel:** `plantmetwiki-rdf`

**Endpoint:** set `SPARQL_ENDPOINT` below (default: local Virtuoso).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from SPARQLWrapper import SPARQLWrapper, CSV
import io

SPARQL_ENDPOINT = "https://sparql-plantmetwiki.bioinformatics.nl/sparql"
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

def run_query(query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT against the endpoint and return a DataFrame."""
    sw = SPARQLWrapper(SPARQL_ENDPOINT)
    sw.setReturnFormat(CSV)
    sw.setQuery(query)
    result = sw.query().convert()
    return pd.read_csv(io.BytesIO(result))

def save_csv(df: pd.DataFrame, name: str) -> Path:
    path = FIGURES_DIR / name
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows → {path}")
    return path

def save_fig(fig: plt.Figure, name: str) -> None:
    for ext in ("pdf", "svg"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}", bbox_inches="tight")
    print(f"Saved → {FIGURES_DIR}/{name}.{{pdf,svg}}")

print(f"Endpoint: {SPARQL_ENDPOINT}")
print(f"Figures dir: {FIGURES_DIR.resolve()}")

---
## 1. Fetch data via SPARQL

Each cell runs one query and saves a CSV. Skip any cell and load the reference CSV instead if the endpoint is unavailable.

In [ ]:
# Genes (GeneProduct DataNodes) per pathway
genes_df = run_query("""
PREFIX wp:    <http://vocabularies.wikipathways.org/wp#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?node a wp:GeneProduct ;
          wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
""")
save_csv(genes_df, "genes_per_pathway.csv")
genes_df.head(3)

In [ ]:
# Metabolites (Metabolite DataNodes) per pathway
metabolites_df = run_query("""
PREFIX wp:    <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?node a wp:Metabolite ;
          wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
""")
save_csv(metabolites_df, "metabolites_per_pathway.csv")
metabolites_df.head(3)

In [ ]:
# Enzymes (Protein DataNodes) per pathway
enzymes_df = run_query("""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?node) AS ?count)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?node a wp:Protein ;
          wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
""")
save_csv(enzymes_df, "enzymes_per_pathway.csv")
enzymes_df.head(3)

In [ ]:
# Biochemical conversions per pathway
conversions_df = run_query("""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?pwID ?title (COUNT(DISTINCT ?interaction) AS ?count)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?interaction a wp:Conversion ;
                 wp:isPartOf ?pwID .
    ?pwID rdfs:label ?title .
  }
}
GROUP BY ?pwID ?title
ORDER BY DESC(?count)
""")
save_csv(conversions_df, "conversions_per_pathway.csv")
conversions_df.head(3)

In [ ]:
# Species (distinct taxa) per pathway via taxonomy-extra graph
species_df = run_query("""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi: <http://purl.obolibrary.org/obo/NCBITaxon_>

SELECT ?pwID (COUNT(DISTINCT ?species) AS ?count)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra> {
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?node wp:isPartOf ?pwID .
  }
}
GROUP BY ?pwID
ORDER BY DESC(?count)
""")
save_csv(species_df, "species_per_pathway.csv")
species_df.head(3)

In [ ]:
# Interaction type counts
interaction_types_df = run_query("""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>

SELECT ?type (COUNT(?interaction) AS ?n)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?interaction a ?type .
    FILTER(STRSTARTS(STR(?type), "http://vocabularies.wikipathways.org/wp#"))
    FILTER(?interaction a wp:Interaction || ?interaction a wp:DirectedInteraction ||
           ?interaction a wp:Conversion  || ?interaction a wp:Catalysis ||
           ?interaction a wp:Inhibition  || ?interaction a wp:Stimulation)
  }
}
GROUP BY ?type
ORDER BY DESC(?n)
""")
save_csv(interaction_types_df, "interaction_types.csv")
interaction_types_df

In [ ]:
# Per-species metrics
per_species_df = run_query("""
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?species
       (COUNT(DISTINCT ?pathway) AS ?pathways)
       (COUNT(DISTINCT ?gene)    AS ?genes)
       (COUNT(DISTINCT ?protein) AS ?enzymes)
       (COUNT(DISTINCT ?met)     AS ?metabolites)
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra> {
    ?node wp:organism ?species .
    FILTER(?species != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?node wp:isPartOf ?pathway .
    OPTIONAL { ?gene a wp:GeneProduct ; wp:isPartOf ?pathway . }
    OPTIONAL { ?protein a wp:Protein  ; wp:isPartOf ?pathway . }
    OPTIONAL { ?met a wp:Metabolite   ; wp:isPartOf ?pathway . }
  }
}
GROUP BY ?species
ORDER BY DESC(?pathways)
""")
save_csv(per_species_df, "per_species_nrs.csv")
per_species_df.head(5)

In [ ]:
# Pathway titles (for scatter plot labels)
titles_df = run_query("""
PREFIX wp:   <http://vocabularies.wikipathways.org/wp#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?pwID ?title
WHERE {
  GRAPH <http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways> {
    ?pwID a wp:Pathway ;
          rdfs:label ?title .
  }
}
""")
save_csv(titles_df, "pathway_titles.csv")
print(f"{len(titles_df):,} pathway titles")

---
## Load CSVs (use reference data if endpoint unavailable)

If any query above failed, load the reference CSVs that were exported from a previous Snorql run:

In [ ]:
def load_csv(name):
    return pd.read_csv(FIGURES_DIR / name)

def find_col(df, needle):
    needle = needle.lower()
    for c in df.columns:
        if needle in c.lower():
            return c
    raise ValueError(f"No column containing '{needle}'. Columns: {list(df.columns)}")

def load_counts(name, count_name="count"):
    df = load_csv(name)
    cc = find_col(df, "count")
    df = df.rename(columns={cc: count_name})
    df[count_name] = pd.to_numeric(df[count_name], errors="coerce").fillna(0)
    try:
        pwid_col = find_col(df, "pwid")
        df = df.rename(columns={pwid_col: "pwID"})
        df["pwID"] = df["pwID"].astype(str)
    except ValueError:
        pass
    try:
        tc = find_col(df, "title")
        df = df.rename(columns={tc: "title"})
    except ValueError:
        pass
    return df

genes       = load_counts("genes_per_pathway.csv",       "genes")
metabolites = load_counts("metabolites_per_pathway.csv", "metabolites")
enzymes     = load_counts("enzymes_per_pathway.csv",     "enzymes")
conversions = load_counts("conversions_per_pathway.csv", "conversions")
species_pw  = load_counts("species_per_pathway.csv",     "species")
int_types   = load_csv("interaction_types.csv")
per_species = load_csv("per_species_nrs.csv")
titles      = load_csv("pathway_titles.csv")

print("Loaded all CSVs ✅")
print(f"  Pathways with gene data:      {len(genes):,}")
print(f"  Pathways with metabolite data:{len(metabolites):,}")
print(f"  Species in per_species table: {len(per_species):,}")

---
## 2. Figure: Overview bar chart (log scale)

Total counts of pathways, genes, metabolites, interactions.

In [ ]:
type_col = find_col(int_types, "type")
n_col    = find_col(int_types, "n") if "n" in int_types.columns else find_col(int_types, "count")
int_types = int_types.rename(columns={type_col: "type", n_col: "n"})
int_types["n"] = pd.to_numeric(int_types["n"], errors="coerce").fillna(0).astype(int)

WP_INTERACTION = "http://vocabularies.wikipathways.org/wp#Interaction"
total_interactions = int(int_types.loc[int_types["type"] == WP_INTERACTION, "n"].iloc[0])

overview = {
    "Pathways":      len(genes),
    "Genes":         int(genes["genes"].sum()),
    "Metabolites":   int(metabolites["metabolites"].sum()),
    "Enzymes":       int(enzymes["enzymes"].sum()),
    "Interactions":  total_interactions,
}

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(list(overview.keys()), list(overview.values()))
ax.set_yscale("log")
ax.set_ylabel("Count (log scale)")
ax.set_title("PlantMetWiki — content overview")
for bar, val in zip(bars, overview.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_overview_barlog")
plt.show()

---
## 3. Figure: Cumulative coverage curves

In [ ]:
def cumulative_curve(df, col):
    s = df[col].sort_values(ascending=False)
    total = s.sum()
    return (s.cumsum() / total).values if total > 0 else s.cumsum().values

curves = {
    "Genes":          cumulative_curve(genes, "genes"),
    "Enzymes":        cumulative_curve(enzymes, "enzymes"),
    "Metabolites":    cumulative_curve(metabolites, "metabolites"),
    "Conversions":    cumulative_curve(conversions, "conversions"),
    "Species/pathway": cumulative_curve(species_pw, "species"),
}

styles = {
    "Genes":           dict(linestyle="-",  marker="o", markevery=100),
    "Enzymes":         dict(linestyle="--", marker="s", markevery=100),
    "Metabolites":     dict(linestyle="-.", marker="^", markevery=100),
    "Conversions":     dict(linestyle=":",  marker="x", markevery=100),
    "Species/pathway": dict(linestyle="--", marker="D", markevery=100),
}

fig, ax = plt.subplots(figsize=(7, 4.8))
for label, curve in curves.items():
    ax.plot(range(1, len(curve)+1), curve, label=label, linewidth=2, **styles[label])
ax.set_xlabel("Pathways (ranked by contribution)")
ax.set_ylabel("Cumulative fraction of total")
ax.set_ylim(0, 1.01)
ax.set_title("Cumulative coverage of PlantMetWiki content")
ax.legend()
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_cumulative_coverage")
plt.show()

---
## 4. Figure: Interaction types

In [ ]:
label_map = {
    "http://vocabularies.wikipathways.org/wp#DirectedInteraction": "Directed interaction",
    "http://vocabularies.wikipathways.org/wp#Conversion":          "Biochemical conversion",
    "http://vocabularies.wikipathways.org/wp#Catalysis":           "Catalysis",
    "http://vocabularies.wikipathways.org/wp#TranscriptionTranslation": "Transcription/translation",
    "http://vocabularies.wikipathways.org/wp#Inhibition":          "Inhibition",
    "http://vocabularies.wikipathways.org/wp#Stimulation":         "Stimulation",
    "http://vocabularies.wikipathways.org/wp#ComplexBinding":      "Complex binding",
    "http://vocabularies.wikipathways.org/wp#Binding":             "Binding",
}

sub = int_types[int_types["type"] != WP_INTERACTION].copy()
sub["label"] = sub["type"].map(label_map).fillna(sub["type"])
sub["pct"]   = 100 * sub["n"] / total_interactions
sub = sub.sort_values("n", ascending=True)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 4.2),
                                gridspec_kw={"width_ratios": [1, 3]})
ax0.bar(["Total\ninteractions"], [total_interactions])
ax0.set_ylabel("Count")
ax0.set_title("A")
ax0.text(0, total_interactions, f"{total_interactions:,}",
         ha="center", va="bottom", fontsize=10)
ax0.spines[["top","right"]].set_visible(False)

ax1.barh(sub["label"], sub["n"])
ax1.set_xlabel("Number of interactions")
ax1.set_title("B")
xmax = sub["n"].max()
for y, (n, pct) in enumerate(zip(sub["n"], sub["pct"])):
    ax1.text(n + xmax * 0.01, y, f"{n:,}  ({pct:.1f}%)", va="center", fontsize=9)
ax1.set_xlim(0, xmax * 1.25)
ax1.spines[["top","right"]].set_visible(False)

fig.suptitle("Interaction types in PlantMetWiki", y=1.02)
plt.tight_layout()
save_fig(fig, "plantmetwiki_interaction_types")
plt.show()

---
## 5. Figure: Species metrics — stacked bar (top 50 species)

In [ ]:
per_species.columns = [c.strip().strip('"') for c in per_species.columns]
for c in ["pathways","genes","enzymes","metabolites"]:
    if c in per_species.columns:
        per_species[c] = pd.to_numeric(per_species[c], errors="coerce").fillna(0)

species_col = find_col(per_species, "species")
per_species = per_species.rename(columns={species_col: "species"})

# Shorten long URIs to just the taxon ID for readability
per_species["species"] = per_species["species"].apply(
    lambda x: x.split("/")[-1].replace("NCBITaxon_","ncbi:") if x.startswith("http") else x
)

top50 = per_species.sort_values("pathways", ascending=False).head(50).copy()
stack_metrics = [c for c in ["genes","enzymes","metabolites"] if c in top50.columns]

COLORS = {"genes": "C1", "enzymes": "C2", "metabolites": "C3"}

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(top50))
bottom = np.zeros(len(top50))
for m in stack_metrics:
    ax.bar(x, top50[m].values, bottom=bottom, label=m.capitalize(), color=COLORS[m])
    bottom += top50[m].values
ax.set_xticks(x)
ax.set_xticklabels(top50["species"], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Count")
ax.set_title("PlantMetWiki content by species (top 50)")
ax.legend(ncol=3, frameon=False)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_species_metrics_stacked_top50_absolute")
plt.show()

---
## 6. Figure: Scatter — genes vs metabolites per pathway (sized by species)

In [ ]:
merged = metabolites[["pwID","metabolites"]].merge(
    genes[["pwID","genes"]], on="pwID", how="outer"
).merge(
    species_pw[["pwID","species"]], on="pwID", how="outer"
).fillna(0)

# Add titles
title_col = find_col(titles, "title")
pwid_col  = find_col(titles, "pwid")
titles2   = titles.rename(columns={pwid_col: "pwID", title_col: "title"})
merged = merged.merge(titles2[["pwID","title"]], on="pwID", how="left")
merged = merged[(merged["metabolites"] > 0) | (merged["genes"] > 0)].copy()

sp_min, sp_max = merged["species"].min(), merged["species"].max()
size = 6 + (merged["species"] - sp_min) / max(sp_max - sp_min, 1) * 154

bins   = [0, 1, 2, 5, 10, int(sp_max)]
bins   = sorted(set(bins))
blabels = []
for i in range(1, len(bins)):
    lo = bins[i-1]+1 if i > 1 else 0
    hi = bins[i]
    blabels.append(f"{lo}-{hi}" if lo != hi else str(hi))
merged["sp_bin"] = pd.cut(merged["species"], bins=bins, include_lowest=True, right=True, labels=blabels)
merged["sp_code"] = merged["sp_bin"].cat.codes

fig, ax = plt.subplots(figsize=(6.8, 5.4))
sc = ax.scatter(merged["metabolites"], merged["genes"],
                s=size, c=merged["sp_code"], cmap="viridis", alpha=0.35, edgecolors="none")
ax.set_xlabel("Metabolites per pathway")
ax.set_ylabel("Genes per pathway")
ax.set_title("Pathway content in PlantMetWiki")

cmap = plt.get_cmap("viridis")
norm = mpl.colors.Normalize(vmin=0, vmax=max(1, len(blabels)-1))
handles = [Line2D([0],[0], marker="o", linestyle="None",
                  markerfacecolor=cmap(norm(i)), markeredgecolor="none",
                  markersize=8, alpha=0.8)
           for i in range(len(blabels))]
ax.legend(handles, blabels, title="Species count", loc="best", frameon=False)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "scatter_genes_vs_metabolites_size_species")
plt.show()

# Save outlier table
merged.sort_values(["genes","metabolites"], ascending=False)[
    ["pwID","title","genes","metabolites","species"]
].to_csv(FIGURES_DIR / "top_pathways_by_genes.csv", index=False)
print("Saved top_pathways_by_genes.csv")

---
## Sandbox — add new queries and figures here

In [ ]:
# Write new SPARQL query or figure here
pass